# TN4 — Domain shift: ánh sáng, nhiễu, vị trí (Google Colab)

**Câu hỏi:** model train trong điều kiện chuẩn; khi điều kiện chụp thay đổi, mỗi phương pháp suy giảm ra sao, và có "sụp đổ" theo những cách khác nhau không?

| Phần | Dữ liệu | Shift |
|---|---|---|
| **TN4a** | MVTec AD 2: `vial`, `sheet_metal` | Ánh sáng thật: cùng một cảnh chụp ở 6–7 điều kiện |
| **TN4b** | MVTec AD: `screw`, `grid`, `transistor` | Nhân tạo: độ sáng, nhiễu, làm mờ, xoay, dịch |

Đánh giá ở hai mức: không cần ngưỡng (I-AUROC, P-AUROC, AUPRO@0.3, AUPRO@0.05) và **tại ngưỡng cố định** (trung bình + 3 độ lệch chuẩn trên ảnh good validation, như paper MVTec AD 2): FPR, TPR, SegF1. Thường gặp trường hợp AUROC vẫn cao nhưng ngưỡng đã hỏng hẳn.

`vial` dùng 256×192, `sheet_metal` 256×1024 để giữ tỷ lệ ảnh gốc (lỗi sheet_metal chỉ khoảng 0,3% diện tích). AE có đầu vào cố định 256×256: ảnh được resize trước khi vào AE, anomaly map phóng về kích thước gốc. Thời gian trên T4: TN4a 25–35 phút, TN4b 30–40 phút, xoay/dịch bản sửa 30–45 phút.

## 1. Cài đặt
Kết nối Drive, lấy code từ repo (nhánh `an/tn-refactor`) và kiểm tra GPU. Sau một lỗi CUDA phải *Disconnect and delete runtime* rồi chạy lại từ đầu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/AnDpTri/mvtec-anomaly-detection.git'
BRANCH = 'an/tn-refactor'
if subprocess.run(['git', '-C', '/content/repo', 'pull', '-q'], capture_output=True).returncode != 0:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'], check=True)
sys.path.insert(0, '/content/repo')

from src.tn.config import check_gpu
check_gpu()

## 2. Cấu hình

In [ ]:
from pathlib import Path
from src.tn.results import Results

DRIVE_ROOT = Path('/content/drive/MyDrive/MVtecAD_Deep_Learning')
DATA_ROOT  = Path('/content/data/MVTec_AD')                 # chép ra ổ local của Colab (đọc nhanh hơn Drive)
RESULTS    = DRIVE_ROOT / 'results'
MAPS_DIR   = RESULTS / 'maps'
FIG_DIR    = RESULTS / 'figures' / 'tn4'
for p in (RESULTS, MAPS_DIR, FIG_DIR): p.mkdir(parents=True, exist_ok=True)
res = Results(RESULTS)
AD2_ROOT   = Path('/content/data/MVTec_AD_2')

METHODS    = ['padim', 'patchcore', 'ae']
CATEGORIES = ['screw', 'grid', 'transistor']        # TN4b

AD2_CATS    = {'vial': (256, 192),                  # (cao, rộng); ảnh gốc 1400x1900
               'sheet_metal': (256, 1024)}          # ảnh gốc 4224x1056
AD2_CORESET = 0.01                                  # bank rất lớn -> coreset 1% (như paper PatchCore)

SYN_SHIFTS = {'brightness': [0.6, 0.8, 1.2, 1.4],   # nhân cường độ
              'noise':      [5, 10, 20],            # độ lệch chuẩn nhiễu Gauss (thang 0-255)
              'blur':       [1, 2, 3],              # sigma (pixel)
              'rotate':     [5, 10, 20],            # độ
              'translate':  [8, 16, 32]}            # pixel (trên ảnh 256)
SYN_GEOM_OVERRIDE = {'transistor': {'rotate': [2, 5], 'translate': [4, 8]}}   # dịch/xoay lớn chính là lỗi 'misplaced'

## 3. Dữ liệu
MVTec AD từ `mvtec_3cat.zip` trên Drive; MVTec AD 2 tải thẳng từ trang MVTec (~2,3 GB).

In [ ]:
from src.tn import data as D, download
download.prepare_mvtec(DATA_ROOT, CATEGORIES, DRIVE_ROOT / 'mvtec_3cat.zip')
download.prepare_ad2(AD2_ROOT, AD2_CATS, DRIVE_ROOT / 'mvtec_ad2_2cat.zip')

## 4. TN4a: MVTec AD 2, ánh sáng thật
Train một lần trên `train/good` (ánh sáng thường), ngưỡng từ `validation/good`, test trên `test_public` và tách theo điều kiện. Mỗi điều kiện chỉ có 4–5 ảnh good và 15 ảnh lỗi: FPR/TPR tại ngưỡng và dòng `all` đáng tin hơn AUROC từng điều kiện.

In [ ]:
from src.tn.experiments import run_tn4a, run_tn4b, run_tn4b_geo
run_tn4a(res, AD2_ROOT, AD2_CATS, METHODS, AD2_CORESET, MAPS_DIR)

## 5. TN4b: shift nhân tạo trên MVTec AD
Train trên 85% ảnh train good, ngưỡng trên 15% còn lại, rồi áp từng shift lên toàn bộ ảnh test (mask đi theo biến đổi hình học).

In [ ]:
run_tn4b(res, DATA_ROOT, CATEGORIES, METHODS, SYN_SHIFTS, SYN_GEOM_OVERRIDE)

## 6. TN4b: xoay/dịch làm lại đúng cách
Bản ở mục 5 xoay/dịch ngay trên ảnh 256 px nên lẫn hai hiệu ứng phụ: viền được lấp tạo texture lạ, và nội suy làm ảnh mờ. Bản này xoay/dịch ở độ phân giải gốc rồi mới thu về 256, loại vùng viền khỏi chấm điểm (co thêm 32 px vì trường tiếp nhận của CNN), và bỏ ảnh lỗi mà vùng lỗi còn dưới 50% trong vùng hợp lệ. Mỗi phương pháp train một lần; bắt buộc có GPU. Phần phân tích tự dùng kết quả này thay cho bản cũ.

In [ ]:
run_tn4b_geo(res, DATA_ROOT, CATEGORIES, METHODS, SYN_SHIFTS, SYN_GEOM_OVERRIDE)

## 7. Phân tích TN4a

In [ ]:
import pandas as pd
from src.tn import analysis as A

a2 = res.load('tn4_ad2')
cols = ['i_auroc', 'p_auroc', 'aupro', 'aupro_005', 'img_fpr_thr', 'img_tpr_thr', 'pix_f1_thr']
for cat in a2.category.unique():
    t = a2[a2.category == cat].pivot_table(index='cond', columns='method', values=cols)
    print(f'\n===== {cat} ====='); display(t.reindex([c for c in A.COND_ORDER + ['all'] if c in t.index]).round(3))
a2.to_csv(RESULTS / 'tn4_ad2_summary.csv', index=False)

sc = pd.read_csv(RESULTS / 'tn4_ad2_scores.csv').drop_duplicates(['category', 'method', 'scene', 'cond'], keep='last')
sc['ratio'] = sc.score / sc.thr_img
for cat in a2.category.unique():
    A.plot_conditions(a2, cat, FIG_DIR / f'tn4a_{cat}_conditions.png')
    A.plot_score_shift(sc, cat, FIG_DIR / f'tn4a_{cat}_score_shift.png')

In [ ]:
for cat, hw in AD2_CATS.items():               # cùng MỘT cảnh lỗi ở mọi điều kiện ánh sáng
    thr = a2[(a2.category == cat) & (a2.cond == 'all')].set_index('method')['thr_img'].to_dict()
    A.plot_same_scene(AD2_ROOT, cat, hw, MAPS_DIR, thr, FIG_DIR / f'tn4a_{cat}_same_scene.png')

## 8. Phân tích TN4b

In [ ]:
sy = A.merge_geo(res.load('tn4_syn'), res.load('tn4_syn_geo2'))
sy.to_csv(RESULTS / 'tn4_syn_summary.csv', index=False)
kinds = [k for k in SYN_SHIFTS if k in set(sy.kind)]
for cat in sy.category.unique():
    A.plot_shifts(sy, cat, kinds, FIG_DIR / f'tn4b_{cat}.png')

summ = A.worst_case(sy)                          # mức shift mạnh nhất so với không shift
summ.to_csv(RESULTS / 'tn4b_worst_case.csv', index=False)
display(summ.pivot_table(index=['kind', 'category'], columns='method', values=['d_i_auroc', 'fpr_thr']))